# 03 — Hub Fitness Index

**Project:** Where Should Amazon Air Build Its Next Hub?
**Author:** Aikerim Imanbayeva

## What this notebook does

Builds the **Hub Fitness Index** — a weighted composite score that ranks candidate airports on five public-data dimensions. The top-scoring airports become the recommended hub locations.

## The five dimensions

| # | Dimension | Source | Why it matters |
| --- | --- | --- | --- |
| 1 | **Current cargo activity** | BTS T-100 | Proven cargo infrastructure; existing operations to absorb |
| 2 | **Runway capacity** | FAA | Can the airport physically operate 767-300F at full payload |
| 3 | **Population catchment** | Census + TIGER | Last-mile reach — drives delivery time savings |
| 4 | **Weather reliability** | NOAA (placeholder for now) | Operational uptime; cargo flies at night when weather is worst |
| 5 | **Geographic complementarity** | Distance to existing Amazon Air | Network gap-filling; new hubs adjacent to existing ones add little |

## Methodology

1. Compute a raw metric per airport for each dimension
2. Normalize each dimension to a 0–100 score via min-max scaling
3. Combine via weighted average → final Hub Fitness Index
4. Run sensitivity analysis on the weights to confirm rankings are robust

## Output files (saved to `data/processed/`)

- `hub_fitness_scores.csv` — all 80 airports with all 5 dimension scores + final index
- `top3_recommendations.csv` — the headline answer to Amazon's question
- `sensitivity_rankings.csv` — top 10 under each weighting scheme

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RAW = PROJECT_ROOT / 'data' / 'raw'
DASH = PROJECT_ROOT / 'dashboards'
DASH.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

AMAZON_ORANGE = '#FF9900'
NEUTRAL = '#4A4A4A'
ACCENT = '#007EB9'
MUTED = '#BBBBBB'

print('Setup complete')

In [ ]:
# Load processed data from notebooks 01
top80 = pd.read_csv(PROCESSED / 'top_cargo_airports.csv')
counties = pd.read_csv(PROCESSED / 'county_population_with_centroids.csv')
amazon = pd.read_csv(RAW / 'amazon_air' / 'amazon_air_facilities.csv')

print(f'Top-80 cargo airports: {len(top80):,}')
print(f'Counties with centroids: {len(counties):,}')
print(f'Amazon Air facilities: {len(amazon):,}')

## Helper: haversine distance

Great-circle distance between two lat/lon points, in miles. Used for both:
- Population catchment (county centroids → airport)
- Geographic complementarity (existing Amazon hubs → airport)

In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    """Vectorized great-circle distance in miles. Inputs may be scalars or arrays."""
    R_MILES = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R_MILES * np.arcsin(np.sqrt(a))

# Sanity check: CVG (Cincinnati) → LAL (Lakeland, FL) should be ~830 miles
d = haversine_miles(39.05, -84.67, 27.99, -82.02)
print(f'CVG → LAL: {d:.0f} miles (expected ~830)')

---
## Dimension 1: Current cargo activity

**Raw metric:** 2025 total throughput (lifted + received, in lbs)

**Normalization:** log-min-max. We use log scaling because cargo volumes span 4 orders of magnitude (SDF handles 700× more than the smallest top-80 airport). Linear scaling would crush everyone below the top 5 to near zero.

**Defense:** *"I used log-scaling for cargo because the distribution is highly skewed — linear scaling would collapse all but the top 5 into a single score band, killing the ranking signal among real candidates."*

In [ ]:
df = top80.copy()

def minmax01(x):
    return (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x))

# Use log1p to handle airports with zero cargo gracefully (none in top80 but defensive)
df['raw_cargo'] = df['total_throughput_lb']
df['score_cargo'] = 100 * minmax01(np.log1p(df['raw_cargo']))

print('Cargo score — top 10:')
print(df.nlargest(10, 'score_cargo')[['ident','name','raw_cargo','score_cargo']]
      .assign(raw_cargo_M=lambda d: (d['raw_cargo']/1e6).round(0)).drop(columns='raw_cargo')
      .to_string(index=False))

---
## Dimension 2: Runway capacity

**Raw metric:** weighted combination of (a) longest runway length and (b) number of runways ≥ 7,000 ft.

The longest runway tells you the upper bound of what aircraft can operate. The count of long runways tells you about parallel-ops capacity (handle two simultaneous heavy aircraft = more throughput per day).

**Weighting:** 70% longest runway, 30% count. Longest runway is the binding physical constraint; count is a throughput multiplier.

In [ ]:
df['score_runway_length'] = 100 * minmax01(df['longest_runway_ft'])
df['score_runway_count'] = 100 * minmax01(df['n_runways_over_7000ft'].clip(upper=4))  # diminishing returns above 4
df['score_runway'] = 0.7 * df['score_runway_length'] + 0.3 * df['score_runway_count']

print('Runway score — top 10:')
print(df.nlargest(10, 'score_runway')[['ident','name','longest_runway_ft','n_runways_over_7000ft','score_runway']]
      .to_string(index=False))

---
## Dimension 3: Population catchment

**Raw metric:** total population in counties whose centroid is within 240 miles (≈4-hour drive proxy) of the airport.

**Method:** For each airport, vectorized haversine to every county centroid; filter to within 240 mi; sum populations.

**Defensible limitation:** straight-line distance ignores real geography (mountains, no highways). For the final report, top-3 candidates will get verified with real drive-time isochrones.

In [ ]:
CATCHMENT_RADIUS_MI = 240

# Vectorize over airports — for each, compute distances to all counties at once
ap_lats = df['lat'].values[:, None]    # shape (n_airports, 1)
ap_lons = df['lon'].values[:, None]
co_lats = counties['centroid_lat'].values[None, :]  # shape (1, n_counties)
co_lons = counties['centroid_lon'].values[None, :]
co_pops = counties['population'].values

dist_mat = haversine_miles(ap_lats, ap_lons, co_lats, co_lons)
in_radius = dist_mat <= CATCHMENT_RADIUS_MI
df['raw_catchment_pop'] = (in_radius * co_pops).sum(axis=1)
df['score_catchment'] = 100 * minmax01(df['raw_catchment_pop'])

print(f'Catchment computed (radius = {CATCHMENT_RADIUS_MI} miles)')
print('\nCatchment score — top 10:')
print(df.nlargest(10, 'score_catchment')[['ident','name','state','raw_catchment_pop','score_catchment']]
      .assign(catchment_M=lambda d: (d['raw_catchment_pop']/1e6).round(1)).drop(columns='raw_catchment_pop')
      .to_string(index=False))

---
## Dimension 4: Weather reliability — PLACEHOLDER

**Status:** Real NOAA Climate Normals data has not yet been pulled. This dimension uses a **latitude-based proxy** as a stand-in.

**Heuristic:** lower latitude → milder weather → higher reliability. This is well-supported empirically — northern US airports (BOS, MSP, ORD, PIT) average 25+ disruption days/year vs. <5 for southern airports (MIA, LAX, PHX) — but it's not perfect (it misses fog at SFO, thunderstorms in FL).

**TODO:** Once NOAA Climate Data Online API token is acquired, replace this with real disruption-day metrics (snow days, low-visibility days, thunderstorm days).

In [ ]:
# Latitude proxy: rescale so southernmost = 100, northernmost = 0
df['raw_weather_lat'] = df['lat']
df['score_weather'] = 100 * (1 - minmax01(df['lat']))

print('Weather score (latitude proxy) — top 10 (warmest, most reliable):')
print(df.nlargest(10, 'score_weather')[['ident','name','state','lat','score_weather']]
      .to_string(index=False))
print('\nWeather score — bottom 5 (coldest, most weather risk):')
print(df.nsmallest(5, 'score_weather')[['ident','name','state','lat','score_weather']]
      .to_string(index=False))

---
## Dimension 5: Geographic complementarity

**Raw metric:** distance (miles) from candidate airport to the nearest existing Amazon Air facility.

**Logic:** a new hub adjacent to CVG/SBD/LAL/ILN adds near-zero network value (overlapping coverage). A new hub 1,500 miles from any existing facility fills a real gap.

**Cap at 1,500 miles:** beyond that, additional distance doesn't add marginal value (you've already filled the gap). Without this cap, Hawaii would dominate the score.

**Interview defense:** *"This is the dimension that turns 'rank airports' into 'design a network.' Without it, we'd just recommend re-investing where Amazon already operates."*

In [ ]:
COMPLEMENTARITY_CAP_MI = 1500

# Only count Amazon Air sortation hubs and regional hubs for complementarity
# (Gateways/mini-stations are smaller and don't really define network coverage)
hubs = amazon[amazon['role'].isin(['primary_hub', 'regional_hub'])].copy()
print(f'Using {len(hubs)} Amazon Air hubs for complementarity:')
print(hubs[['airport_id','role','city','state']].to_string(index=False))

# For each candidate, compute distance to each hub, take the minimum
hub_lats = hubs['latitude'].values[None, :]
hub_lons = hubs['longitude'].values[None, :]
ap_lats = df['lat'].values[:, None]
ap_lons = df['lon'].values[:, None]

dist_to_hubs = haversine_miles(ap_lats, ap_lons, hub_lats, hub_lons)
df['raw_min_dist_to_amazon_hub_mi'] = dist_to_hubs.min(axis=1)
df['raw_capped_complementarity'] = df['raw_min_dist_to_amazon_hub_mi'].clip(upper=COMPLEMENTARITY_CAP_MI)
df['score_complementarity'] = 100 * minmax01(df['raw_capped_complementarity'])

print('\nComplementarity score — top 10 (furthest from existing hubs = biggest gap fill):')
print(df.nlargest(10, 'score_complementarity')[
    ['ident','name','state','raw_min_dist_to_amazon_hub_mi','score_complementarity']
].round(0).to_string(index=False))

---
## Hub Fitness Index — default weighting (equal)

Default weights: **20% per dimension** — interview-defensible as a neutral starting point. The sensitivity analysis below tests how rankings change under different weight schemes.

In [ ]:
DEFAULT_WEIGHTS = {
    'score_cargo':         0.20,
    'score_runway':        0.20,
    'score_catchment':     0.20,
    'score_weather':       0.20,
    'score_complementarity': 0.20,
}

def compute_hub_index(df, weights):
    return sum(df[col] * w for col, w in weights.items())

df['hub_fitness_index'] = compute_hub_index(df, DEFAULT_WEIGHTS)
df = df.sort_values('hub_fitness_index', ascending=False).reset_index(drop=True)
df['fitness_rank'] = df.index + 1

print('Hub Fitness Index — TOP 15 (equal weights):')
cols = ['fitness_rank','ident','name','state','is_amazon_air',
        'score_cargo','score_runway','score_catchment','score_weather','score_complementarity',
        'hub_fitness_index']
print(df.head(15)[cols].round(1).to_string(index=False))

---
## Top recommendations — excluding airports Amazon already operates

The question is *where to build NEXT*, so we filter to candidates **without** existing Amazon Air presence.

In [ ]:
candidates_only = df[~df['is_amazon_air']].copy().reset_index(drop=True)
candidates_only['recommendation_rank'] = candidates_only.index + 1

top3 = candidates_only.head(3).copy()
print('=' * 90)
print('TOP 3 RECOMMENDED AIRPORTS FOR AMAZON AIR\'S NEXT REGIONAL HUB')
print('=' * 90)
print()
for _, row in top3.iterrows():
    print(f"  #{row['recommendation_rank']}: {row['ident']} — {row['name']} ({row['state']})")
    print(f"       Hub Fitness Index: {row['hub_fitness_index']:.1f}")
    print(f"       Cargo: {row['score_cargo']:.0f}  Runway: {row['score_runway']:.0f}  "
          f"Catchment: {row['score_catchment']:.0f}  Weather: {row['score_weather']:.0f}  "
          f"Complementarity: {row['score_complementarity']:.0f}")
    print(f"       Distance to nearest Amazon hub: {row['raw_min_dist_to_amazon_hub_mi']:.0f} miles")
    print()

---
## Visualization 1 — Top 15 scores

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
top15 = candidates_only.head(15).iloc[::-1]  # reverse for bar order
colors = [AMAZON_ORANGE if r <= 3 else ACCENT for r in top15['recommendation_rank']]
ax.barh(top15['ident'], top15['hub_fitness_index'], color=colors)
ax.set_xlabel('Hub Fitness Index (0–100, weighted average)')
ax.set_title('Top 15 Recommended Hub Locations (excluding airports Amazon already operates)',
             loc='left', fontsize=13, pad=12)
for i, (_, row) in enumerate(top15.iterrows()):
    ax.text(row['hub_fitness_index'] + 0.6, i,
            f"  {row['hub_fitness_index']:.1f} — {row['name'][:35]}",
            va='center', fontsize=9, color=NEUTRAL)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig(DASH / 'top15_hub_fitness.png', bbox_inches='tight')
plt.show()

---
## Visualization 2 — Dimension heatmap for top 15

In [ ]:
score_cols = ['score_cargo','score_runway','score_catchment','score_weather','score_complementarity']
score_names = ['Cargo','Runway','Catchment','Weather','Complementarity']
heatmap_data = candidates_only.head(15).set_index('ident')[score_cols]
heatmap_data.columns = score_names

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd', vmin=0, vmax=100,
            cbar_kws={'label':'Dimension score'}, ax=ax, linewidths=0.5, linecolor='white')
ax.set_xlabel('Dimension')
ax.set_ylabel('')
ax.set_title('Dimension Scores — Top 15 Candidate Airports', loc='left', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(DASH / 'top15_dimension_heatmap.png', bbox_inches='tight')
plt.show()

---
## Visualization 3 — Map of top 3 recommendations + existing Amazon footprint

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

# Background: all top-80 cargo airports
ax.scatter(top80['lon'], top80['lat'], s=20, color=MUTED, alpha=0.4,
           label='Top-80 cargo airports')

# Existing Amazon Air
ax.scatter(amazon['longitude'], amazon['latitude'], s=80, color=AMAZON_ORANGE,
           marker='^', edgecolor='black', linewidth=0.5,
           label='Existing Amazon Air facilities', zorder=3)

# Top 3 recommendations
ax.scatter(top3['lon'], top3['lat'], s=350, color=ACCENT, marker='*',
           edgecolor='black', linewidth=1, label='Top 3 recommended hubs', zorder=4)
for _, r in top3.iterrows():
    ax.annotate(f"#{r['recommendation_rank']} {r['ident']}",
                (r['lon'], r['lat']),
                textcoords='offset points', xytext=(12, 8),
                fontsize=12, fontweight='bold', color=ACCENT)

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Top 3 Recommended Hub Locations vs. Existing Amazon Air Footprint',
             loc='left', fontsize=13, pad=12)
ax.set_xlim(-130, -65); ax.set_ylim(23, 50)
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(DASH / 'top3_recommendations_map.png', bbox_inches='tight')
plt.show()

---
## Sensitivity analysis — how robust are the top 3 to different weighting schemes?

Test the top recommendations under four alternative weight schemes:

| Scheme | Cargo | Runway | Catchment | Weather | Complementarity | Philosophy |
| --- | --- | --- | --- | --- | --- | --- |
| **Equal** (default) | 20% | 20% | 20% | 20% | 20% | Neutral starting point |
| **Catchment-heavy** | 15% | 10% | 40% | 15% | 20% | "Reach drives the business" |
| **Operations-heavy** | 25% | 25% | 15% | 25% | 10% | "Build where we can operate reliably" |
| **Gap-filling** | 10% | 15% | 25% | 15% | 35% | "Maximize geographic coverage" |

In [ ]:
schemes = {
    'Equal':            {'score_cargo':.20,'score_runway':.20,'score_catchment':.20,'score_weather':.20,'score_complementarity':.20},
    'Catchment-heavy':  {'score_cargo':.15,'score_runway':.10,'score_catchment':.40,'score_weather':.15,'score_complementarity':.20},
    'Operations-heavy': {'score_cargo':.25,'score_runway':.25,'score_catchment':.15,'score_weather':.25,'score_complementarity':.10},
    'Gap-filling':      {'score_cargo':.10,'score_runway':.15,'score_catchment':.25,'score_weather':.15,'score_complementarity':.35},
}

sens_results = {}
for name, w in schemes.items():
    tmp = candidates_only.copy()
    tmp['idx'] = compute_hub_index(tmp, w)
    tmp = tmp.sort_values('idx', ascending=False).reset_index(drop=True)
    sens_results[name] = tmp[['ident','name','state','idx']].head(10).reset_index(drop=True)
    sens_results[name].columns = [f'{c}_{name}' if c=='ident' else c for c in sens_results[name].columns]

# Side-by-side table of top 10 under each scheme
sens_table = pd.DataFrame({name: df_['ident_'+name].values for name, df_ in sens_results.items()})
sens_table.index = sens_table.index + 1
sens_table.index.name = 'rank'
print('Top 10 candidates under each weighting scheme:')
print(sens_table.to_string())

In [ ]:
# Robustness: how often does each airport appear in top 10 across all schemes?
all_top10 = pd.concat([df_[[f'ident_{n}']].rename(columns={f'ident_{n}':'ident'}) for n, df_ in sens_results.items()])
robustness = all_top10['ident'].value_counts().rename('top10_appearances')
robust_picks = robustness[robustness >= 3]
print(f'Airports appearing in top 10 in 3+ of 4 schemes (most robust picks):')
print(robust_picks.to_string())

**Interpretation:**

The robust picks — airports that show up in the top 10 across most weighting schemes — are the most defensible recommendations. An airport that's #1 under one scheme but #25 under another is fragile; an airport that's #2–#7 across all schemes is genuinely strong.

---
## Save outputs

In [ ]:
# Final scoring table — all 80 airports, all 5 dimensions + final index
out_cols = ['fitness_rank','ident','name','state','is_amazon_air',
            'score_cargo','score_runway','score_catchment','score_weather','score_complementarity',
            'hub_fitness_index',
            'raw_cargo','longest_runway_ft','raw_catchment_pop','lat','raw_min_dist_to_amazon_hub_mi']
df[out_cols].to_csv(PROCESSED / 'hub_fitness_scores.csv', index=False)

# Top 3 with all detail (the headline deliverable)
top3.to_csv(PROCESSED / 'top3_recommendations.csv', index=False)

# Sensitivity table
sens_table.to_csv(PROCESSED / 'sensitivity_rankings.csv')

print('Saved:')
for f in ['hub_fitness_scores.csv','top3_recommendations.csv','sensitivity_rankings.csv']:
    p = PROCESSED / f
    print(f'  {f:35s}  {p.stat().st_size/1024:>7.1f} KB')

---
## Summary & next steps

**What this notebook produced:**

- Hub Fitness Index scores for all 80 top-cargo candidate airports across 5 dimensions
- Headline answer: top 3 recommended hubs (excluding airports Amazon already operates)
- Sensitivity analysis confirming robustness across 4 different weighting philosophies
- 3 visualizations saved to `dashboards/`

**Important caveat — weather dimension is a placeholder.**

The weather score currently uses a latitude proxy. For production-quality analysis, this should be replaced with real NOAA disruption-day data (snow days, low-visibility days, thunderstorm days) for each candidate airport. That data pull is the next task.

**Limitations to note in the final report:**

1. Straight-line catchment distance — real drive-time isochrones would refine the top 3
2. Latitude weather proxy — real NOAA data needed for production
3. Equal weighting is a starting point; the right weights depend on Amazon's strategic priorities
4. Excludes cost-of-living and labor-market factors (real-world considerations for hub site selection)

**Next deliverables:**

1. NOAA weather pull (replaces placeholder)
2. Business memo (.docx or .pdf) summarizing the recommendation
3. Interactive storyboard HTML
4. Publish to GitHub with README